# placement: is each piece centered on its square? (full design, perception)

README "Perception and rules": *verify physical placement: piece centered on its square
within tolerance*. This is separate from rule legality (`board_assembly.ipynb`) and from the
engine comparison (`board_sync.ipynb`). The robot needs it after its own moves (did the
gripper drop the piece off-center?) and before grasping (where exactly is the piece?).

Two parts:
1. **Detection** (TODO): estimate where a piece's base touches the board, in warp pixels.
   Use the base, not the silhouette center: tall pieces lean in perspective.
2. **Tolerance check** (implemented): convert that point to board meters with the warp
   geometry from `homography.ipynb`, subtract the square center, compare with a tolerance.

Milestone: full design (beyond the class project M1-M5).

**TODO**
- [ ] `estimate_piece_base`: choose a method (e.g. segment the piece against the known square
      color in the warped image and take the lowest blob point / fitted base ellipse; or a
      detector bbox bottom-center from `yolo.ipynb`)
- [ ] choose `tolerance_m` from the gripper's capture range (how far off-center can a piece
      be and still be grasped?), maybe separate x/y tolerances
- [ ] correct for the piece height: a point seen above the board plane maps to the wrong
      place under a board-plane homography

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import chess
import numpy as np

%run ./homography.ipynb

## Offsets and the tolerance check
Offset = (piece base in board meters) - (square center in board meters), so it is in
meters along the files (`du`, toward h) and ranks (`dv`, toward 8), independent of the warp
resolution and view.

In [ ]:
DEFAULT_TOLERANCE_FRAC = 0.2   # TODO: placeholder, 20% of a square (5 mm on the 25 mm board)


@dataclass
class PlacementCheck:
    square: str
    offset_m: tuple[float, float]   # (du, dv) in meters
    distance_m: float
    ok: bool


def placement_offset(sq: int, base_warp_xy, geom: BoardGeometry, warp) -> np.ndarray:
    """(du, dv) meters from the center of square sq to a point given in warp pixels."""
    uv = warp_to_board(np.asarray(base_warp_xy, float), geom, warp)
    return uv - square_center_board(sq, geom)


def check_placement(sq: int, base_warp_xy, geom: BoardGeometry, warp,
                    tolerance_m: float | None = None) -> PlacementCheck:
    """Is the piece on square sq within tolerance_m (Euclidean) of the square center?"""
    tol = DEFAULT_TOLERANCE_FRAC * geom.square if tolerance_m is None else tolerance_m
    d = placement_offset(sq, base_warp_xy, geom, warp)
    dist = float(np.hypot(*d))
    return PlacementCheck(chess.square_name(sq), (float(d[0]), float(d[1])), dist, dist <= tol)


def check_all(bases: dict[int, tuple[float, float]], geom: BoardGeometry, warp,
              tolerance_m: float | None = None) -> list[PlacementCheck]:
    """Check every occupied square; bases maps square index -> base point in warp pixels."""
    return [check_placement(sq, xy, geom, warp, tolerance_m) for sq, xy in sorted(bases.items())]


def which_square(base_warp_xy, geom: BoardGeometry, warp) -> int | None:
    """Square index containing the point, or None if it is off the playing area."""
    u, v = warp_to_board(np.asarray(base_warp_xy, float), geom, warp) / geom.square
    if not (0 <= u < 8 and 0 <= v < 8):
        return None
    return chess.square(int(u), int(v))

## Detection (TODO)

In [ ]:
def estimate_piece_base(warped: np.ndarray, sq: int, geom: BoardGeometry, warp) -> tuple[float, float] | None:
    """TODO: base point (x, y) in warp pixels of the piece on square sq, or None if empty.

    Inputs: the warped top-down BGR image (homography.ipynb: warp_board) and a square index.
    Look in a window around the square (the piece may be off-center by up to ~half a square).
    Hints: compare with the expected square color (light/dark) to get a piece mask; the base
    is the part of the mask closest to the camera (bottom of the black_bottom warp); fit a
    circle/ellipse to the bottom contour. Validate in sim, where true positions are known.
    """
    raise NotImplementedError("TODO: estimate_piece_base")

## Self-check

In [ ]:
from types import SimpleNamespace

_pl_geom = load_board_geometry()
_pl_w = SimpleNamespace(square_px=100, margin_px=100, view="black_bottom")
_e4c = board_to_warp(square_center_board(chess.E4, _pl_geom), _pl_geom, _pl_w)
_chk = check_placement(chess.E4, _e4c, _pl_geom, _pl_w)
assert _chk.ok and _chk.distance_m < 1e-9 and _chk.square == "e4"
# 4 mm toward the h-file is within 5 mm; 6 mm is not
_off4 = board_to_warp(square_center_board(chess.E4, _pl_geom) + [0.004, 0], _pl_geom, _pl_w)
_off6 = board_to_warp(square_center_board(chess.E4, _pl_geom) + [0, 0.006], _pl_geom, _pl_w)
assert check_placement(chess.E4, _off4, _pl_geom, _pl_w).ok
_c6 = check_placement(chess.E4, _off6, _pl_geom, _pl_w)
assert not _c6.ok and np.allclose(_c6.offset_m, (0, 0.006))
assert which_square(_off6, _pl_geom, _pl_w) == chess.E4
assert which_square([5, 5], _pl_geom, _pl_w) is None
assert [c.ok for c in check_all({chess.E4: _off4, chess.D4: _off6}, _pl_geom, _pl_w)] == [False, True]  # d4 first
print("placement OK")